# Beyond Pandas
Building Interactive Data Analytics Applications

Run each cell in order. The same sales dataset is used throughout.

## Install libraries

In [ ]:
%pip install -q pandas matplotlib plotly streamlit

## Import Pandas and load data

In [ ]:
import pandas as pd
df = pd.read_csv("sales_data.csv")
df.head()

## Dataset dimensions

In [ ]:
print(df.shape)

## Column names

In [ ]:
print(df.columns)

## Data types and completeness

In [ ]:
df.info()

## Summary statistics

In [ ]:
df.describe()

## Select columns

In [ ]:
small = df[["Region", "Sales"]]
small.head()

## Filter records

In [ ]:
south = df[df["Region"] == "South"]
south.head()

## Group and summarize

In [ ]:
region_sales = (df.groupby("Region", as_index=False)["Sales"].sum().sort_values("Sales", ascending=False))
print(region_sales)

## Regional sales chart

In [ ]:
import matplotlib.pyplot as plt
plt.bar(region_sales["Region"], region_sales["Sales"])
plt.title("Sales by Region")
plt.ylabel("Sales (₹)")
plt.show()

## Matplotlib category comparison

In [ ]:
category_sales = df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
category_sales.plot(kind="bar")
plt.title("Total Sales by Category")
plt.xlabel("Category")
plt.ylabel("Sales (₹)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Plotly monthly trend

In [ ]:
import plotly.express as px
df["Date"] = pd.to_datetime(df["Date"])
monthly = (df.assign(Month=df["Date"].dt.to_period("M").astype(str)).groupby("Month", as_index=False)["Sales"].sum())
fig = px.line(monthly, x="Month", y="Sales", markers=True, title="Monthly Sales Trend")
fig.show()

## Plotly category comparison

In [ ]:
fig1 = px.bar(df.groupby("Category", as_index=False)["Sales"].sum(), x="Category", y="Sales", color="Category", title="Sales by Category")
fig1.show()

## Plotly scatter chart

In [ ]:
fig2 = px.scatter(df, x="Quantity", y="Sales", color="Region", hover_data=["Product", "Category"], title="Quantity vs Sales")
fig2.show()

## Plotly sample dataset: iris

In [ ]:
px.data.iris().head()

## Plotly sample dataset: tips

In [ ]:
px.data.tips().head()

## Plotly sample dataset: gapminder

In [ ]:
px.data.gapminder().head()

## Plotly sample dataset: wind

In [ ]:
px.data.wind().head()

## Streamlit application setup
Each task writes a Python script and launches a real Streamlit server. The preview is interactive. Only one server runs at a time. If the inline preview is blocked, open the displayed local link.

In [ ]:
import sys, subprocess, time, urllib.request
from pathlib import Path
from IPython.display import display, IFrame, HTML
server = globals().get("server")
PORT = 8501

def stop_app():
    global server
    if server is not None and server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=5)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()

def run_app(filename):
    global server
    stop_app()
    with open("streamlit_workshop.log", "w") as log:
        server = subprocess.Popen([sys.executable, "-m", "streamlit", "run", filename,
            "--server.port", str(PORT), "--server.headless", "true"], stdout=log, stderr=subprocess.STDOUT)
    for _ in range(60):
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/_stcore/health", timeout=1) as response:
                if response.status == 200:
                    break
        except OSError:
            time.sleep(0.5)
    else:
        raise RuntimeError(Path("streamlit_workshop.log").read_text())
    url = f"http://localhost:{PORT}"
    display(HTML(f'<a href="{url}" target="_blank">Open Streamlit application</a>'))
    display(IFrame(url, width="100%", height=650))

## First Streamlit application

In [ ]:
%%writefile hello_app.py
import streamlit as st
st.set_page_config(page_title="My First App")
st.title("My First Streamlit App")
st.header("Hello, Data Analytics!")
st.write("We are building an interactive application.")

In [ ]:
run_app('hello_app.py')

## Interactive widgets

In [ ]:
%%writefile widgets_app.py
import streamlit as st
region = st.selectbox("Choose a region", ["All", "North", "South", "East", "West"])
show_details = st.checkbox("Show details")
quantity_limit = st.slider("Minimum quantity", 1, 6, 1)
st.write("Selected region:", region)
st.write("Minimum quantity:", quantity_limit)
if show_details:
    st.info("Additional details are visible.")

In [ ]:
run_app('widgets_app.py')

## Dashboard step 1: Load data and create dashboard shell

In [ ]:
%%writefile dashboard_step_1.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

In [ ]:
run_app('dashboard_step_1.py')

## Dashboard step 2: Add sidebar filters

In [ ]:
%%writefile dashboard_step_2.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

In [ ]:
run_app('dashboard_step_2.py')

## Dashboard step 3: Filter records

In [ ]:
%%writefile dashboard_step_3.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

filtered = df.copy()
if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]
if selected_category != "All":
    filtered = filtered[filtered["Category"] == selected_category]
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = filtered[filtered["Date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]

if filtered.empty:
    st.warning("No records match these filters. Please change your selection.")
    st.stop()


st.write("Matching records:", len(filtered))
st.dataframe(filtered)

In [ ]:
run_app('dashboard_step_3.py')

## Dashboard step 4: Calculate KPI metrics

In [ ]:
%%writefile dashboard_step_4.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

filtered = df.copy()
if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]
if selected_category != "All":
    filtered = filtered[filtered["Category"] == selected_category]
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = filtered[filtered["Date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]

if filtered.empty:
    st.warning("No records match these filters. Please change your selection.")
    st.stop()

# KPI metrics
total_sales = filtered["Sales"].sum()
total_orders = len(filtered)
average_order = filtered["Sales"].mean()
units_sold = filtered["Quantity"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total sales", f"₹{total_sales:,.0f}")
c2.metric("Transactions", f"{total_orders:,}")
c3.metric("Average transaction", f"₹{average_order:,.0f}")
c4.metric("Units sold", f"{units_sold:,}")

In [ ]:
run_app('dashboard_step_4.py')

## Dashboard step 5: Category comparison chart

In [ ]:
%%writefile dashboard_step_5.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

filtered = df.copy()
if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]
if selected_category != "All":
    filtered = filtered[filtered["Category"] == selected_category]
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = filtered[filtered["Date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]

if filtered.empty:
    st.warning("No records match these filters. Please change your selection.")
    st.stop()

# KPI metrics
total_sales = filtered["Sales"].sum()
total_orders = len(filtered)
average_order = filtered["Sales"].mean()
units_sold = filtered["Quantity"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total sales", f"₹{total_sales:,.0f}")
c2.metric("Transactions", f"{total_orders:,}")
c3.metric("Average transaction", f"₹{average_order:,.0f}")
c4.metric("Units sold", f"{units_sold:,}")

# Charts
left, right = st.columns(2)
with left:
    by_category = filtered.groupby("Category", as_index=False)["Sales"].sum()
    fig_bar = px.bar(by_category, x="Category", y="Sales", color="Category",
                     title="Sales by category", text_auto=".2s")
    st.plotly_chart(fig_bar, use_container_width=True)

In [ ]:
run_app('dashboard_step_5.py')

## Dashboard step 6: Monthly sales trend

In [ ]:
%%writefile dashboard_step_6.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

filtered = df.copy()
if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]
if selected_category != "All":
    filtered = filtered[filtered["Category"] == selected_category]
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = filtered[filtered["Date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]

if filtered.empty:
    st.warning("No records match these filters. Please change your selection.")
    st.stop()

# KPI metrics
total_sales = filtered["Sales"].sum()
total_orders = len(filtered)
average_order = filtered["Sales"].mean()
units_sold = filtered["Quantity"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total sales", f"₹{total_sales:,.0f}")
c2.metric("Transactions", f"{total_orders:,}")
c3.metric("Average transaction", f"₹{average_order:,.0f}")
c4.metric("Units sold", f"{units_sold:,}")

# Charts
left, right = st.columns(2)
with left:
    by_category = filtered.groupby("Category", as_index=False)["Sales"].sum()
    fig_bar = px.bar(by_category, x="Category", y="Sales", color="Category",
                     title="Sales by category", text_auto=".2s")
    st.plotly_chart(fig_bar, use_container_width=True)

with right:
    monthly = (filtered.assign(Month=filtered["Date"].dt.to_period("M").astype(str))
               .groupby("Month", as_index=False)["Sales"].sum())
    fig_line = px.line(monthly, x="Month", y="Sales", markers=True,
                       title="Monthly sales trend")
    st.plotly_chart(fig_line, use_container_width=True)

In [ ]:
run_app('dashboard_step_6.py')

## Dashboard step 7: Scatter chart and transaction table

In [ ]:
%%writefile dashboard_step_7.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

filtered = df.copy()
if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]
if selected_category != "All":
    filtered = filtered[filtered["Category"] == selected_category]
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = filtered[filtered["Date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]

if filtered.empty:
    st.warning("No records match these filters. Please change your selection.")
    st.stop()

# KPI metrics
total_sales = filtered["Sales"].sum()
total_orders = len(filtered)
average_order = filtered["Sales"].mean()
units_sold = filtered["Quantity"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total sales", f"₹{total_sales:,.0f}")
c2.metric("Transactions", f"{total_orders:,}")
c3.metric("Average transaction", f"₹{average_order:,.0f}")
c4.metric("Units sold", f"{units_sold:,}")

# Charts
left, right = st.columns(2)
with left:
    by_category = filtered.groupby("Category", as_index=False)["Sales"].sum()
    fig_bar = px.bar(by_category, x="Category", y="Sales", color="Category",
                     title="Sales by category", text_auto=".2s")
    st.plotly_chart(fig_bar, use_container_width=True)

with right:
    monthly = (filtered.assign(Month=filtered["Date"].dt.to_period("M").astype(str))
               .groupby("Month", as_index=False)["Sales"].sum())
    fig_line = px.line(monthly, x="Month", y="Sales", markers=True,
                       title="Monthly sales trend")
    st.plotly_chart(fig_line, use_container_width=True)

fig_scatter = px.scatter(filtered, x="Quantity", y="Sales", color="Region",
                         hover_data=["Product", "Category", "Date"],
                         title="Quantity vs sales")
st.plotly_chart(fig_scatter, use_container_width=True)

st.subheader("Filtered transaction records")
st.dataframe(filtered.sort_values("Date", ascending=False), use_container_width=True)

In [ ]:
run_app('dashboard_step_7.py')

## Dashboard step 8: Download filtered data

In [ ]:
%%writefile dashboard_step_8.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

filtered = df.copy()
if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]
if selected_category != "All":
    filtered = filtered[filtered["Category"] == selected_category]
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = filtered[filtered["Date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]

if filtered.empty:
    st.warning("No records match these filters. Please change your selection.")
    st.stop()

# KPI metrics
total_sales = filtered["Sales"].sum()
total_orders = len(filtered)
average_order = filtered["Sales"].mean()
units_sold = filtered["Quantity"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total sales", f"₹{total_sales:,.0f}")
c2.metric("Transactions", f"{total_orders:,}")
c3.metric("Average transaction", f"₹{average_order:,.0f}")
c4.metric("Units sold", f"{units_sold:,}")

# Charts
left, right = st.columns(2)
with left:
    by_category = filtered.groupby("Category", as_index=False)["Sales"].sum()
    fig_bar = px.bar(by_category, x="Category", y="Sales", color="Category",
                     title="Sales by category", text_auto=".2s")
    st.plotly_chart(fig_bar, use_container_width=True)

with right:
    monthly = (filtered.assign(Month=filtered["Date"].dt.to_period("M").astype(str))
               .groupby("Month", as_index=False)["Sales"].sum())
    fig_line = px.line(monthly, x="Month", y="Sales", markers=True,
                       title="Monthly sales trend")
    st.plotly_chart(fig_line, use_container_width=True)

fig_scatter = px.scatter(filtered, x="Quantity", y="Sales", color="Region",
                         hover_data=["Product", "Category", "Date"],
                         title="Quantity vs sales")
st.plotly_chart(fig_scatter, use_container_width=True)

st.subheader("Filtered transaction records")
st.dataframe(filtered.sort_values("Date", ascending=False), use_container_width=True)

csv = filtered.to_csv(index=False).encode("utf-8")
st.download_button("Download filtered data (CSV)", data=csv,
                   file_name="filtered_sales_data.csv", mime="text/csv")

In [ ]:
run_app('dashboard_step_8.py')

## Complete interactive sales dashboard

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(page_title="Sales Analytics Dashboard", page_icon="📊", layout="wide")

@st.cache_data
def load_data():
    return pd.read_csv("sales_data.csv", parse_dates=["Date"])

df = load_data()

st.title("📊 Interactive Sales Analytics Dashboard")
st.caption("Explore sales performance by region, category and time.")

# Sidebar filters
st.sidebar.header("Filter the data")
regions = ["All"] + sorted(df["Region"].unique().tolist())
categories = ["All"] + sorted(df["Category"].unique().tolist())
selected_region = st.sidebar.selectbox("Choose region", regions)
selected_category = st.sidebar.selectbox("Choose category", categories)
min_date = df["Date"].min().date()
max_date = df["Date"].max().date()
date_range = st.sidebar.date_input("Choose date range", (min_date, max_date))

filtered = df.copy()
if selected_region != "All":
    filtered = filtered[filtered["Region"] == selected_region]
if selected_category != "All":
    filtered = filtered[filtered["Category"] == selected_category]
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
    filtered = filtered[filtered["Date"].between(pd.Timestamp(start_date), pd.Timestamp(end_date))]

if filtered.empty:
    st.warning("No records match these filters. Please change your selection.")
    st.stop()

# KPI metrics
total_sales = filtered["Sales"].sum()
total_orders = len(filtered)
average_order = filtered["Sales"].mean()
units_sold = filtered["Quantity"].sum()

c1, c2, c3, c4 = st.columns(4)
c1.metric("Total sales", f"₹{total_sales:,.0f}")
c2.metric("Transactions", f"{total_orders:,}")
c3.metric("Average transaction", f"₹{average_order:,.0f}")
c4.metric("Units sold", f"{units_sold:,}")

# Charts
left, right = st.columns(2)
with left:
    by_category = filtered.groupby("Category", as_index=False)["Sales"].sum()
    fig_bar = px.bar(by_category, x="Category", y="Sales", color="Category",
                     title="Sales by category", text_auto=".2s")
    st.plotly_chart(fig_bar, use_container_width=True)

with right:
    monthly = (filtered.assign(Month=filtered["Date"].dt.to_period("M").astype(str))
               .groupby("Month", as_index=False)["Sales"].sum())
    fig_line = px.line(monthly, x="Month", y="Sales", markers=True,
                       title="Monthly sales trend")
    st.plotly_chart(fig_line, use_container_width=True)

fig_scatter = px.scatter(filtered, x="Quantity", y="Sales", color="Region",
                         hover_data=["Product", "Category", "Date"],
                         title="Quantity vs sales")
st.plotly_chart(fig_scatter, use_container_width=True)

st.subheader("Filtered transaction records")
st.dataframe(filtered.sort_values("Date", ascending=False), use_container_width=True)

csv = filtered.to_csv(index=False).encode("utf-8")
st.download_button("Download filtered data (CSV)", data=csv,
                   file_name="filtered_sales_data.csv", mime="text/csv")

In [ ]:
run_app('app.py')

## Stop Streamlit server

In [ ]:
stop_app()